# HypDiff on Google Colab

Run the **HypDiff** project (Hyperbolic Diffusion for graph generation) on Colab with a free GPU.

This notebook keeps your project as a real Python package (it does **not** flatten your `.py` files into cells), so all the cross-module imports (`config`, `diff`, `graph_train`, `manifolds/`, `layers/`, `models/`, …) keep working exactly as they do on your machine.

**How to use it**
1. `Runtime → Change runtime type → GPU` (T4 is fine).
2. Run the cells top to bottom.
3. Pick **one** of the two "Get the code" options below (Google Drive *or* GitHub).

---

## 0. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Get the code — choose ONE option

### Option A — Google Drive (recommended)
Upload your whole `HypDiff` folder to Google Drive (e.g. to `MyDrive/HypDiff`), then run this cell.
This is the most reliable option because it brings **everything** — including the trained checkpoint
`final_trained_model.pt` and the `data/` folder — with no GitHub auth needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this to wherever you uploaded the folder in your Drive:
PROJECT_DIR = '/content/drive/MyDrive/HypDiff'

import os
assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR} — upload the HypDiff folder to Drive and fix this path."
%cd $PROJECT_DIR
print("Working in:", os.getcwd())
print(sorted(os.listdir())[:20])

### Option B — clone from GitHub (alternative)
Use this instead of Option A if your repo is public (or you have a token for a private repo).
The checkpoint is stored with **Git LFS**, so we install `git-lfs` first.

> If the repo is **private**, replace the URL with
> `https://<YOUR_TOKEN>@github.com/IritKogan/HypDiff.git`

In [ ]:
# !apt-get -qq install git-lfs >/dev/null && git lfs install
# !git clone https://github.com/IritKogan/HypDiff.git /content/HypDiff
# %cd /content/HypDiff
# !git lfs pull        # download final_trained_model.pt
# import os; print(os.getcwd()); print(sorted(os.listdir())[:20])

## 2. Install dependencies

Your `requirements.txt` is a full `pip freeze` from your Windows machine (torch 2.12, numpy 2.4, tensorflow, …).
Installing that verbatim would fight with the versions Colab ships and can break CUDA.
Instead we keep Colab's own PyTorch and add only the extra libraries HypDiff needs on top:
**geoopt** (manifold-aware tensors), **torch-geometric**, and **dgl**.

If a later cell raises `ModuleNotFoundError: X`, just `!pip install X` and re-run.

In [ ]:
# Extras that Colab doesn't ship by default:
!pip install -q geoopt

# torch-geometric (matched to Colab's installed torch/CUDA):
!pip install -q torch-geometric

# DGL — pick the wheel matching Colab's CUDA (see https://www.dgl.ai/pages/start.html if this fails):
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html || pip install -q dgl

# Sanity check
import geoopt, torch_geometric
print("geoopt", geoopt.__version__, "| torch_geometric", torch_geometric.__version__)

## 3. Environment variables

`main.py` expects these two env vars (it sets them itself, but we set them here too so the
package also works when imported directly rather than run as a script).

In [ ]:
import os
os.environ['DATAPATH'] = 'data/'
os.environ.setdefault('LOG_DIR', 'logs/')
os.makedirs('logs', exist_ok=True)

## 4. Run HypDiff

`main.py` is driven by command-line flags (defined in `config.py`). The key ones:

| flag | meaning | options |
|---|---|---|
| `--taskselect` | which pipeline | `lptask`, `graphtask` |
| `--type` | train or evaluate | `train`, `test` |
| `--dataset` | dataset | e.g. `MUTAG` |
| `--cuda` | GPU device (‑1 = CPU) | `0` |
| `--epochs`, `--epoch_diff` | training length | ints |

### 4a. Quick test with the trained checkpoint (fast)

In [ ]:
!python main.py --taskselect lptask --type test --dataset MUTAG --cuda 0

### 4b. Train from scratch (slow — needs a GPU runtime)

In [ ]:
# Link-prediction / diffusion task:
# !python main.py --taskselect lptask --type train --dataset MUTAG --cuda 0 --epochs 200

# Or the graph-generation task:
# !python main.py --taskselect graphtask --dataset MUTAG --cuda 0

See every available flag and its default:

In [ ]:
!python main.py --help

## 5. Perturbation / Stage-1 scripts (optional)

Your project also has standalone scripts (`run_perturbation.py`, `test_perturbation.py`,
`stage0_smoke_test.py`). Run them the same way once the folder is the working directory:

In [ ]:
# !python run_perturbation.py
# !python stage0_smoke_test.py

---
### Notes
- **Outputs** (generated graphs, plots, logs) are written under the working directory
  (`generated_images/`, `logs/`, `log/`). On **Option A** these save straight back into your Google Drive.
- If you hit a numpy/torch ABI error, restart the runtime (`Runtime → Restart`) after the install cell and re-run from step 2.
